# L4: Real-time Inference with the KV-cache

In this lesson, you will cache a fixed training context and measure how much server prediction work it saves across repeated requests. You will compare `TabPFNClassifier`'s default `fit_preprocessors` mode against `fit_with_cache` on a 20,000-row training dataset, verify that both modes produce identical predictions, and benchmark server prediction time alongside whole-request latency across repeated batches.

## 1. Setup


Load the benchmarking helpers and authenticate your `tabpfn_client` session. You will use client `0.6.0` (with TabPFN v3.5) so you can inspect server-side prediction timings alongside the total end-to-end request latency.

In [1]:
import os

print("TabPFN token available:", bool(os.getenv("TABPFN_TOKEN")))

TabPFN token available: True


In [2]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

from tabpfn_client import TabPFNClassifier

from helper import (
    N_ESTIMATORS, RANDOM_STATE,
    benchmark_cache_comparison, configure_lesson,
    init_tabpfn, show_fit_comparison,
    show_latency_comparison, show_split_summary,
    timed_client_call, display_precomputed
)

configure_lesson()
init_tabpfn()

## 2. Generate the data

Create a synthetic binary classification dataset of 30,000 rows with 20 numeric features, splitting it into 20,000 training rows and 10,000 test rows. Using a larger training context makes the benefit of caching much clearer, since the model has more training rows to encode.

In [3]:
X, y = make_classification(
    n_samples=30_000, n_features=20, n_informative=10,
    n_redundant=5, random_state=RANDOM_STATE,
)
X = pd.DataFrame(X, columns=[f"x{i}" for i in range(20)])
y = pd.Series(y, name="target")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=10_000,
    stratify=y, random_state=RANDOM_STATE,
)
show_split_summary(X_train, X_test)
X_train.head(3)

,rows,features
training set,20000,20
test set,10000,20


,x0,x1,x2,...,x17,x18,x19
11657,0.148635,-1.044472,-2.624505,...,-0.516864,-1.059521,2.318312
6265,0.948096,-0.379064,-0.785177,...,3.622826,0.564117,0.717118
13607,-1.947707,-1.259804,0.476027,...,0.590712,0.687446,-1.345450


## 3. No-cache baseline

By default, `TabPFNClassifier` uses `fit_mode="fit_preprocessors"`, which only prepares the data locally. Every time you call `predict_proba`, the server has to re-encode the entire 20,000-row training context from scratch. Let's time both the full API request (`t_predict`) and the actual server prediction time (`predict_s`).

In [4]:
clf = TabPFNClassifier()

_, t_fit = timed_client_call(clf.fit, X_train.values, y_train.values)

print(f"Context ready in {t_fit:.2f} s")

Context ready in 2.34 s


In [5]:
proba, t_predict = timed_client_call(
    clf.predict_proba, X_test.values
)
server_time = clf.last_predict_timings["predict_s"]
proba = proba[:, 1]
auc = roc_auc_score(y_test, proba)

print("For 20K training rows and 10K test rows")
print(f"Server prediction: {server_time:.3f} s")
print(f"Whole request: {t_predict:.2f} s")
print(f"Test AUC: {auc:.3f}")

For 20K training rows and 10K test rows
Server prediction: 3.238 s
Whole request: 4.60 s
Test AUC: 0.996


## 4. Using the KV-cache

Now initialize `TabPFNClassifier` with `fit_mode="fit_with_cache"`. This shifts the heavy lifting to `fit` time, computing and storing the transformer's key-value (KV) states for the training data once on the server. When you call `predict_proba`, the model simply attends to that cached context—saving server compute while producing the exact same predictions.

In [6]:
clf_cached = TabPFNClassifier(fit_mode="fit_with_cache",)

_, t_fit_cached = timed_client_call(
    clf_cached.fit, X_train.values, y_train.values
)
print(f"Context ready in {t_fit_cached:.2f} s")

Context ready in 7.11 s


In [ ]:
proba_cached, t_predict_cached = timed_client_call(
    clf_cached.predict_proba, X_test.values
)
server_time_cached = clf_cached.last_predict_timings["predict_s"]
proba_cached = proba_cached[:, 1]
auc_cached = roc_auc_score(y_test, proba_cached)

print("For 20K training rows and 10K test rows")
print(f"Server prediction: {server_time_cached:.3f} s")
print(f"Whole request: {t_predict_cached:.2f} s")
print(f"Test AUC: {auc_cached:.3f}")

In [ ]:
show_fit_comparison(
    [t_fit, t_fit_cached], [t_predict, t_predict_cached],
    [server_time, server_time_cached], n_rows=len(X_test),
)
difference = np.abs(proba - proba_cached).max()
print(f"Largest probability difference: {difference:.4f}")

## 5. Benchmark repeated requests


A single call isn't enough to measure a reliable speedup, especially when scoring all 10,000 test rows at once. To simulate real-time inference, the benchmark helper warms up both models and then sends 20 repeated requests of 1,000-row batches to each, alternating the order to keep network conditions fair.

We will compare two clocks:

- **Server prediction (`server_ms`):** The time the server worker spent running the model. This is where reusing the cached training context directly saves compute.
- **Whole request (`request_ms`):** The total wall-clock time you waited, which includes data uploads, preprocessing, network round-trips, and queue wait times.

A ratio (`uncached / cached`) above `1.0×` means the KV-cache was faster.

In [ ]:
run_benchmark = True
if run_benchmark:
    timings = benchmark_cache_comparison(
        clf, clf_cached, X_test,
        n_queries=20,
        rows_per_query=1_000,
    )
    show_latency_comparison(timings)
else:
    display_precomputed()

## When not to use the cache

While caching speeds up repeated inference, it isn't always the right choice:

- **Frequently changing training data:** You have to refit and rebuild the cache whenever your training context updates.
- **One-off predictions:** If you only need a single `fit` followed by a single `predict`, the extra time spent building the cache during `fit` won't pay off.
- **Memory constraints:** Keeping a cached KV context alive uses server-side memory.

## Recap

In this lesson, you learned how to speed up repeated TabPFN inference using the KV-cache:

- **Cache a fixed context (`fit_with_cache`):** Encode the training data once during `fit` when you plan to run repeated predictions against the same dataset.
- **Track two clocks:** Compare server prediction time and whole-request latency separately, since network transfers and server queues also affect total wait time.
- **Benchmark fairly:** Use warm-up calls, matched batches, alternating request order, and median latencies to get reliable numbers.
- **Same predictions, faster execution:** Caching changes *when* the training context is processed, without changing the model's predicted probabilities.